## Obiettivi
Per il momento ci concentriamo sugli **algoritmi centralizzati di random sampling**, per poi ricollegarci al mondo distribuito.

Il problema di **random sampling** può essere formulato come segue: dato un dataset $A$ di grandi dimensioni ($|A| \gg 0$, $|A| = n$), vogliamo estrarre un campione $S$ di dimensione molto più piccola ($|S| = s$, con $s \ll n$) in modo che tale campione sia **rappresentativo** del dataset originale. 

Con **rappresentativo** intendiamo che il campione $S$ deve soddisfare precise e rigorose proprietà statistiche. Nel corso di Big Data è stato già affrontato l'algoritmo centralizzato di **Reservoir Sampling**, ma l'analisi è stata superficiale: ci eravamo limitati a studiare la probabilità marginale per cui $$\forall x \in A, \qquad \Pr(x \in S) = \frac{s}{n}$$
ossia per ogni elemento $x$ del dataset originale, la probabilità che $x$ sia incluso nel campione $S$ è esattamente $s/n$ (si parla di probabilità marginale perché stiamo guardando un solo elemento alla volta, ignorando le relazioni con gli altri elementi del sottinsieme $S$).

Il problema della probabilità marginale è che non basta a garantire che il campione $S$ sia davvero uniforme. Infatti questa controlla solo la probabilità che un singolo elemento sia incluso in $S$, ma non dice nulla sulle probabilità congiunte di più elementi che appaiono insieme in $S$. Potrebbe quindi accadere che ogni elemento abbia la giusta probabilità marginale di essere scelto, ma che poi alcuni sottinsiemi di cardinalità $s$ siano più probabili di altri, come si vede nell'esempio seguente:

es. ipotizziamo di avere un dataset $A = \{a, b, c, d\}$ e vogliamo estrarre un campione di dimensione $s = 2$. I Possibili sample in questo caso sono $\{a, b\}$, $\{a, c\}$, $\{a, d\}$, $\{b, c\}$, $\{b, d\}$ e $\{c, d\}$. Se ogni sample fosse uniforme allora ci aspetteremmo che ogni coppia appaia con probabilità $1/6$. Immaginiamo un algoritmo che sceglie solo le coppie $\{a, b\}$ e $\{c, d\}$ con probabilità $1/2$ ciascuna. In questo caso:
$$\Pr(a \in S) = \Pr(b \in S) = \Pr(c \in S) = \Pr(d \in S) = 1/2$$
quindi la probabilità marginale è corretta per ogni documento, però il sample non è uniforme perché quattro coppie hanno probabilità zero di essere estratte $\{a, c\}$, $\{a, d\}$, $\{b, c\}$ e $\{b, d\}$.

## Proprietà da soddisfare
**Per questo motivo vogliamo garantire la proprietà molto forte per cui il sample prodotto dall'algoritmo deve essere uniformemente distribuito su tutti i possibili sample di dimensione $s$**. 

Si scrive formalmente tale proprietà di uniformità come segue. Sia $\mathcal{S}$ la **variabile aleatoria** che rappresenta il sample prodotto dall'algoritmo. Vogliamo che $\mathcal{S}$ soddisfi la seguente proprietà di uniformità:
$$\forall S \subseteq A : |S| = s, \qquad \Pr(\mathcal{S} = S) = \frac{1}{\binom{n}{s}}$$
ossia ogni sottinsieme di cardinalità s deve avere esattamente la stessa probabilità di essere scelto (nota che $\mathcal{S}$ calligrafico è una variabile aleatoria che rappresenta il sample prodotto dall'algoritmo, mentre $S$ rappresenta un sample fisso, deterministico, di dimensione $s$).

Questa proprietà, essendo più forte della probabilità marginale, la implica.

Un'**ulteriore proprietà** statistica che vorremmo garantire riguarda **l'ordinamento degli elementi nel sample**. Finora abbiamo considerato il sample $S$ come un semplice sottinsieme non ordinato del dataset originale $A$, però in diverse applicazioni è utile che gli elementi campionati siano mantenuti in un ordine casuale.

Vorremmo, più precisamente, che fissato un qualunque sottinsieme $S \subseteq A$ di cardinalità $s$, tutte le possibili permutazioni degli elementi di $S$ siano equiprobabili. Formalmente, indicando con $\vec{S} = <x_1, x_2, \ldots, x_s>$ la versione ordinata del sample e con $\mathcal{P}(s)$ l'insieme di tutte le permutazioni di $s$ elementi, richiediamo che:
$$\forall \pi \in \mathcal{P_s}, \qquad \Pr(\vec{S} = \pi (S)\mid\mathcal{S} = S) = \frac{1}{s!}$$

### Algoritmo di FY_Shuffle
L'algoritmo di **FY_Shuffle** serve a risolvere la seconda proprietà. Dato un insieme $A$ di $n$ elementi memorizzati in un array $\vec{A} = \vec{A}[1:n]$, l'algoritmo di **FY_Shuffle** produce in modo iterativo una permutazione u.a.r. $\vec{A_\pi}$ di $A$.

<img src="img/FY_shuffle.png" alt="FY_Shuffle" width="300"/>

In pratica l'algoritmo, per generare il nuovo array random, parte copiando l'array originale in $\vec{A_\pi}$. Dopodiché, ad ogni iterazione $i$ sceglie un indice $j$ a caso tra $1$ e $i$ e scambia $\vec{A_\pi}[i]$ con $\vec{A_\pi}[j]$.

Assumendo che il costo di estrarre un numero j u.a.r. tra $1$ e $i$ sia $O(1)$ (anche se abbiamo visto in BD che in realtà dovrebbe essere $O(\log i)$ bit-level), il costo totale dell'algoritmo è $O(n)$.

Si dimostra che l'algoritmo di **FY_Shuffle** produce una permutazione u.a.r. di $A$.

> **Lemma 2.1**  
> 
> Alla fine di ogni round $i = 1, 2, \ldots, n$ dell'algoritmo di **FY_Shuffle**, il prefisso $\vec{A_\pi}[1:i]$ è una permutazione uniforme random dei primi $i$ elementi dell'array iniziale $\vec{A}$.

> **Dimostrazione**  
> 
> La dimostrazione procede per induzione su $i$.
>
> Fissiamo una qualunque permutazione $<a_1, \ldots, a_i>$ dei primi $i$ elementi di $\vec{A}$, vogliamo calcolare, alla fine del round $i$, la probabilità dell'evento $$\varepsilon_i = \{\vec{A_\pi}[1:i] = <a_1, \ldots, a_i>\}$$
> (ossia la probabilità che data una qualsiasi permutazione dei primi $i$ elementi di $\vec{A}$, alla fine del round $i$ il prefisso $\vec{A_\pi}[1:i]$ sia proprio quella permutazione). Vorremmo ovviamente che tale probabilità fosse $1/i!$.
> 
> - **P.B. $i = 1$**: in tal caso stiamo considerando solo il primo elemento dell'array, che resterà in quella posizione con probabilità 1. Quindi $\Pr(\varepsilon_1) = 1 = \frac{1}{1!}$, pertanto il passo base è verificato.
>
> - **P.I.**: applicando l'ipotesi induttiva fino al passo $i-1$, sappiamo che $$\Pr(\varepsilon_{i-1}) = \Pr(\vec{A_\pi}[1:i-1] = <a_1, \ldots, a_{i-1}>) = \frac{1}{(i-1)!}$$  
> Al passo i-esimo scelgo $j$ in modo u.a.r. in $[i]$. Allora ci sono due possibili casi disgiunti da considerare:
> - **Caso 1**: $j = i$: in tal caso non viene effettuato alcuno scambio, quindi la prima parte del prefisso $\vec{A_\pi}[1:i-1]$ resta invariata. Poiché la scelta di $j$ è del tutto indipendente dalla costruzione del prefisso $\vec{A_\pi}[1:i-1]$:
> $$\Pr(\varepsilon_i) = \Pr(\varepsilon_{i-1}) \cdot \Pr(j = i) = \frac{1}{(i-1)!} \cdot \frac{1}{i} = \frac{1}{i!}$$
> - **Caso 2**: $j = k$ con $k < i$: in tal caso viene effettuato uno scambio tra $\vec{A_\pi}[i]$ e $\vec{A_\pi}[k]$. 
> Supponiamo di voler ottenere alla fine del round $i$ $\vec{A_\pi}[1:i] = <a_1, \ldots, a_i>$.  
>Ma quindi, prima dello scambio, l'array $\vec{A_\pi}$ doveva essere tale che $\vec{A_\pi}[1:i-1] = <a_1, \ldots, a_{k-1}, a_i, a_{k+1}, \ldots, a_{i-1}>$ (ossia $a_i$ doveva essere in posizione $k$).  
>Ma quindi abbiamo determinato una specifica permutazione del prefisso al passo $i-1$, che per ipotesi induttiva ha probabilità $1/(i-1)!$. Poiché di nuovo la scelta di $j$ è del tutto indipendente dalla costruzione del prefisso $\vec{A_\pi}[1:i-1]$, anche in questo caso:
> $$\Pr(\varepsilon_i) = \Pr(\varepsilon_{i-1}) \cdot \Pr(j = k) = \frac{1}{(i-1)!} \cdot \frac{1}{i} = \frac{1}{i!}$$
> $\blacksquare$

### Algoritmo di FY_Sample
Per soddisfare anche la prima proprietà, si definisce l'algoritmo di **FY_Sample**. L'idea dietro l'algoritmo è semplicissima: si esegue anzitutto **FY_Shuffle** per ottenere una permutazione u.a.r. $\vec{A_\pi}$ di $A$, e poi si restituisce come sample i primi $s$ elementi di $\vec{A_\pi}$, ossia $S = \vec{A_\pi}[1:s]$. 

<img src="img/FY_sample.png" alt="FY_Sample" width="300"/>

Chiaramente eseguire soltanto **FY_Shuffle** fino all'elemento $s$ non sarebbe stato sufficiente a garantire la prima proprietà in quanto non avremmo preso in considerazione gli elementi di $\vec{A_\pi}$ a partire da $s+1$ fino a $n$, per questo deve essere eseguito nell'intero array.

> **Lemma 2.2**  
> 
> Sia $\vec{A}[1:n]$ un vettore di n numeri distinti e sia $s \in [n]$. L'algoritmo di **FY_Sample** genera, su input ($\vec{A}$, $s$), un random sample $\mathcal{S}$ la cui distribuzione soddisfa le due seguenti proprietà:
> 1. Inteso come sottoinsieme non ordinato, $\mathcal{S}$ è uniformemente distribuito sullo spazio $\binom{n}{s}$ di tutti i possibili sample di dimensione $s$ (ossia $\forall S \subseteq A : |S| = s, \Pr(\mathcal{S} = S) = 1/\binom{n}{s}$).
> 2. Intendendo $\mathcal{S}$ come vettore $\vec{S}[1:s]$, questo è ordinato in modo u.a.r. rispetto a tutti i possibili $s!$ ordinamenti di $S$ (ossia $\forall \pi \in \mathcal{P}_s, \Pr(\vec{S} = \pi(S)\mid\mathcal{S} = S) = 1/s!$).
>
> **Dimostrazione**  
> 
> **La proprietà 1.** del teorema si dimostra sfruttando la proprietà di **FY_Shuffle** per cui 
> $$
> \forall \pi \in \mathcal{P}_n,
> \qquad
> \Pr(\vec{A}_{\pi} = \pi(A))
> =
> \frac{1}{n!}
> $$
> Fissiamo ora un sottinsieme $S = {x_1, \ldots, x_s}$ di cardinalità $s$, ci chiediamo qual è la probabilità di ottenere esattamente questo sample (in questo caso non stiamo guardando l'ordinamento degli elementi, ma solo il fatto che $S$ sia il sample restituito dall'algoritmo).  
> Casi favorevoli vs Casi possibili: S viene restituito quando i primi s posti della permutazione contengono esattamente gli elementi di S ($x_1, \ldots, x_s$) in qualsiasi posizione (-> $s!$) mentre gli altri $n-s$ elementi di $A$ occupano le rimanenti $n-s$ posizioni sempre in qualsiasi ordine (-> $(n-s)!$).  
> Per quanto riguarda i casi possibili, poiché come detto e dimostrato **FY_Shuffle** restituisce ogni permutazione di $A$ con probabilità $1/n!$, allora i casi possibili sono proprio tutte le permutazioni di $A$ (-> $n!$). Quindi:
> $$
> \Pr(\mathcal{S} = S)
> =
> \frac{s!(n-s)!}{n!}
> =
> \frac{1}{\binom{n}{s}}
> $$
> dimostrando quindi che ogni sample di dimensione $s$ è restituito con la stessa probabilità, ossia che $\mathcal{S}$ è uniformemente distribuito su tutti i possibili sample di dimensione $s$.  
>
> **La proprietà 2.** segue direttamente dalla proprietà di **FY_Shuffle**. Poiché la proprietà di **FY_Shuffle** garantisce che venga generata una permutazione u.a.r. dell'intero array $\vec{A}$, e poiché stiamo considerando i primi $s$ elementi di tale permutazione come parte di $S$ (ossia $S = \vec{A_\pi}[1:s]$), allora è immediato concludere che tutti i loro $s!$ ordinamenti sono equiprobabili.
> 
> $\blacksquare$

## Reservoir Sampling